In [1]:
# ===========================================
# XGBoost with MINIMAL features only:
#   hour, aisle_id, department_id, product_id, user_id  (label: reordered)
# Instacart 6th-order evaluation (order-level F1 with 'None' rule)
# ===========================================
import os, json, gc, time, re, subprocess, warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ---- 0) Colab Drive & Paths ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Colab 외 환경이면 무시:", e)

DATA_DIR = "/content/drive/MyDrive/data/instacart"
MASTER_PATH = os.path.join(DATA_DIR, "master_dataset_with_roles_final.csv")

OUT_DIR = os.path.join(DATA_DIR, "perf_minimal_only6")
os.makedirs(OUT_DIR, exist_ok=True)

# 산출물 경로
OUT_OOF_PATH     = os.path.join(OUT_DIR, "oof_predictions_instacart_minimal.csv")
OUT_DET_TEST     = os.path.join(OUT_DIR, "test_predictions_detailed_minimal.csv")
OUT_ORD_TEST     = os.path.join(OUT_DIR, "test_predictions_orders_minimal_threshold")
OUT_META_PATH    = os.path.join(OUT_DIR, "xgb_minimal_meta.json")
OUT_SUMMARY_CSV  = os.path.join(OUT_DIR, "xgb_minimal_summary.csv")
OUT_SUMMARY_JSON = os.path.join(OUT_DIR, "xgb_minimal_summary.json")
OUT_FEAT_IMP     = os.path.join(OUT_DIR, "xgb_minimal_feature_importance.csv")

# ---- 1) Utils ----
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from typing import List, Optional, Dict, Tuple
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss
import xgboost as xgb

def first_existing(cands: List[str], cols: List[str]) -> Optional[str]:
    for c in cands:
        if c in cols:
            return c
    return None

def is_binary_series(s: pd.Series) -> bool:
    vals = pd.unique(s.dropna())
    return set(vals.tolist()).issubset({0,1}) or set(vals.tolist()).issubset({0.0,1.0})

# Instacart F1: 주문 단위 평균, 빈 예측이나 정답은 'None' 매칭
def f1_single(true_set: set, pred_set: set) -> float:
    if len(true_set) == 0 and len(pred_set) == 0:
        return 1.0
    if len(pred_set) == 0:
        pred_set = {"None"}
    if len(true_set) == 0:
        true_set = {"None"}
    tp = len(true_set & pred_set)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_set)
    recall    = tp / len(true_set)
    return 0.0 if (precision+recall)==0 else 2*precision*recall/(precision+recall)

def order_level_f1(df: pd.DataFrame, order_col: str, product_col: str,
                   target_col: str, proba_col: str, thr: float) -> float:
    f1s = []
    for _, g in df.groupby(order_col):
        true_set = set(g.loc[g[target_col]==1, product_col].tolist())
        pred_set = set(g.loc[g[proba_col]>=thr, product_col].tolist())
        f1s.append(f1_single(true_set, pred_set))
    return float(np.mean(f1s)) if len(f1s) else 0.0

def search_best_threshold_fast(
    df: pd.DataFrame, order_col: str, product_col: str,
    target_col: str, proba_col: str,
    quantile_lo: float = 0.05, quantile_hi: float = 0.95,
    n_candidates: int = 31, max_orders: int = 50_000,
    seed: int = 2025, verbose: bool = True
) -> Tuple[float, float]:
    t0 = time.time()
    orders = df[order_col].drop_duplicates()
    if (max_orders is not None) and (len(orders) > max_orders):
        sampled_orders = orders.sample(max_orders, random_state=seed)
        sub = df[df[order_col].isin(sampled_orders)][[order_col, product_col, target_col, proba_col]].copy()
        if verbose: print(f"[thr-search] sample orders: {len(sampled_orders):,}  rows: {len(sub):,}")
    else:
        sub = df[[order_col, product_col, target_col, proba_col]].copy()
        if verbose: print(f"[thr-search] full orders: {sub[order_col].nunique():,}  rows: {len(sub):,}")
    qs = np.linspace(quantile_lo, quantile_hi, n_candidates)
    thr_list = np.unique(sub[proba_col].quantile(qs).values)
    if verbose: print(f"[thr-search] candidates: {len(thr_list)}  (quantiles {quantile_lo:.2f}~{quantile_hi:.2f})")
    true_cnt = sub.groupby(order_col)[target_col].sum().astype(np.int32)
    best_thr, best_f1 = 0.5, -1.0
    for i, t in enumerate(thr_list, 1):
        mask    = (sub[proba_col] >= t)
        pred_cnt= sub.loc[mask].groupby(order_col, observed=True)[proba_col].size()
        tp_cnt  = sub.loc[mask & (sub[target_col] == 1)].groupby(order_col, observed=True)[target_col].size()
        agg = pd.DataFrame({
            'true': true_cnt,
            'pred': pred_cnt.reindex(true_cnt.index, fill_value=0).astype(np.int32),
            'tp'  : tp_cnt.reindex(true_cnt.index,  fill_value=0).astype(np.int32),
        })
        true = agg['true'].values
        pred = agg['pred'].values
        tp   = agg['tp'].values
        none_case = (pred == 0) & (true == 0)
        with np.errstate(divide='ignore', invalid='ignore'):
            precision = np.divide(tp, pred, out=np.zeros_like(tp, dtype=float), where=pred>0)
            recall    = np.divide(tp, true, out=np.zeros_like(tp, dtype=float), where=true>0)
            denom     = precision + recall
            f1_arr    = np.divide(2*precision*recall, denom, out=np.zeros_like(denom), where=denom>0)
        f1_arr[none_case] = 1.0
        f1 = float(f1_arr.mean())
        if verbose and (i % max(1, len(thr_list)//5) == 0 or i == len(thr_list)):
            print(f"[thr-search] {i}/{len(thr_list)}  thr={t:.4f}  F1={f1:.5f}  elapsed={time.time()-t0:.1f}s")
        if f1 > best_f1:
            best_f1, best_thr = f1, float(t)
    if verbose: print(f"[thr-search] best_thr={best_thr:.4f}  best_f1={best_f1:.5f}")
    return best_thr, best_f1

def fast_auc_logloss(df: pd.DataFrame, y_col: str, p_col: str, max_rows: int = 2_000_000, seed: int = 2025):
    y = df[y_col].to_numpy()
    p = np.clip(df[p_col].to_numpy(dtype=np.float64), 1e-15, 1-1e-15)
    if len(y) > max_rows:
        df = df.sample(n=max_rows, random_state=seed)
        y = df[y_col].to_numpy()
        p = np.clip(df[p_col].to_numpy(dtype=np.float64), 1e-15, 1-1e-15)
        print(f"[metrics] sampled {len(y):,} rows for AUC/Logloss")
    auc = roc_auc_score(y, p) if len(np.unique(y))>1 else np.nan
    ll  = float(log_loss(y, p))
    return auc, ll

def _bst_predict_proba(bst: xgb.Booster, dmat: xgb.DMatrix) -> np.ndarray:
    bi = getattr(bst, "best_iteration", None)
    if bi is not None:
        try:
            return bst.predict(dmat, iteration_range=(0, int(bi)+1)).astype(np.float32)
        except TypeError:
            pass
        try:
            return bst.predict(dmat, ntree_limit=getattr(bst, "best_ntree_limit", int(bi)+1)).astype(np.float32)
        except Exception:
            pass
    return bst.predict(dmat).astype(np.float32)

def detect_device_and_tree_method():
    try:
        _ = subprocess.check_output(["nvidia-smi"])
        return "cuda", "hist"
    except Exception:
        return "cpu", "hist"

DEVICE, TREE_METHOD = detect_device_and_tree_method()
print(f"[INFO] device={DEVICE}  tree_method={TREE_METHOD}  xgboost={xgb.__version__}")

# ---- 2) Load data & split (prior=1~5 train / 6th test) ----
df = pd.read_csv(MASTER_PATH, low_memory=False)
cols = df.columns.tolist()

target_col   = first_existing(['reordered','label','target','y','is_reordered'], cols)
product_col  = first_existing(['product_id','pid','product'], cols)
order_col    = first_existing(['order_id','oid','order'], cols)
member_col   = first_existing(['user_id','member_id','uid','user'], cols)
hour_col     = first_existing(['hour','order_hour_of_day','order_hour','hour_of_day'], cols)
aisle_col    = first_existing(['aisle_id','aisle'], cols)
dept_col     = first_existing(['department_id','dept_id','department'], cols)
eval_col     = 'eval_set' if 'eval_set' in cols else None

assert target_col and product_col and (order_col or member_col), "필수 컬럼 누락: target/product/(order or user)."
assert hour_col,  "hour(or order_hour_of_day) 컬럼이 필요합니다."
assert aisle_col, "aisle_id(또는 유사) 컬럼이 필요합니다."
assert dept_col,  "department_id(또는 유사) 컬럼이 필요합니다."
assert member_col,"user_id(또는 유사) 컬럼이 필요합니다."

# eval_set=='test'(7번째 주문) 제거, prior(1~5) 학습, train(6) 평가
if eval_col:
    ev = df[eval_col].astype(str).str.lower()
    if 'test' in ev.unique():
        before = len(df); df = df.loc[~ev.eq('test')].copy()
        print(f"eval_set=='test' 행 {before - len(df):,}개 제거.")
    mask_train = df[eval_col].astype(str).str.lower().eq('prior')
    mask_test  = df[eval_col].astype(str).str.lower().eq('train')
else:
    mask_train = (df['role_train'].fillna(0).astype(int)==1) if 'role_train' in df.columns else pd.Series([True]*len(df))
    mask_test  = (df['role_test' ].fillna(0).astype(int)==1) if 'role_test'  in df.columns else pd.Series([False]*len(df))

df_train = df.loc[mask_train].copy()
df_test  = df.loc[mask_test].copy()

# 타깃 이진화 보정
if not is_binary_series(df_train[target_col]):
    df_train[target_col] = (df_train[target_col] > 0).astype(np.int8)
if (target_col in df_test.columns) and (not is_binary_series(df_test[target_col])):
    df_test[target_col] = (df_test[target_col] > 0).astype(np.int8)

order_key = order_col if order_col else member_col
group_key = member_col if member_col else order_col

print(f"[INFO] rows — train(prior)={len(df_train):,}, test(6th)={len(df_test):,}")

# ---- 3) Minimal feature set ----
# Only these 5 features as requested (label 'reordered' is NOT a feature)
FEATURES = [hour_col, aisle_col, dept_col, product_col, member_col]
for c in FEATURES:
    df_train[c] = df_train[c].astype('float32')
    df_test[c]  = df_test[c].astype('float32')

# 결측치 보정
df_train[FEATURES] = df_train[FEATURES].fillna(-1.0)
df_test[FEATURES]  = df_test[FEATURES].fillna(-1.0)

print("[INFO] minimal features:", FEATURES)

# ---- 4) (Optional) downsample negatives for speed ----
NEG_POS_RATIO      = 5.0
MAX_TRAIN_ROWS     = 3_000_000
RANDOM_SEED        = 2025
np.random.seed(RANDOM_SEED)

pos_mask = (df_train[target_col]==1)
neg_mask = ~pos_mask
n_pos    = int(pos_mask.sum())
max_neg  = int(min(neg_mask.sum(), NEG_POS_RATIO * n_pos))
neg_idx  = df_train.loc[neg_mask].sample(n=max_neg, random_state=RANDOM_SEED).index if max_neg>0 else df_train.loc[neg_mask].index
train_idx = df_train.loc[pos_mask].index.union(neg_idx)
if len(train_idx) > MAX_TRAIN_ROWS:
    train_idx = pd.Index(np.random.choice(train_idx, size=MAX_TRAIN_ROWS, replace=False))
df_tr_s = df_train.loc[train_idx].copy()
print(f"[INFO] train sample: {len(df_tr_s):,} rows (pos={int((df_tr_s[target_col]==1).sum()):,})")

# ---- 5) Train XGBoost (Booster API, 3.0.4) ----
@dataclass
class XGBCfg:
    n_estimators: int = 600
    max_depth: int = 7
    learning_rate: float = 0.05
    subsample: float = 0.90
    colsample_bytree: float = 0.90
    min_child_weight: float = 1.0
    reg_lambda: float = 1.0
    gamma: float = 0.0
    tree_method: str = "hist"   # use 'hist' + device
    device: str = "cuda" if DEVICE=="cuda" else "cpu"
    random_state: int = 42

cfg = XGBCfg()
NFOLDS = 3
EARLY_STOP = 50

def fit_xgb_cv_booster(df_tr: pd.DataFrame, features: List[str], target: str,
                       groups: Optional[pd.Series], cfg: XGBCfg,
                       model_prefix: str, n_splits: int = NFOLDS,
                       early_stopping_rounds: int = EARLY_STOP) -> Dict:
    X = df_tr[features].values
    y = df_tr[target].values.astype(np.float32)
    if groups is None:
        groups = np.arange(len(y)) % n_splits
    splitter = GroupKFold(n_splits=n_splits)

    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "max_depth": cfg.max_depth,
        "eta": cfg.learning_rate,
        "subsample": cfg.subsample,
        "colsample_bytree": cfg.colsample_bytree,
        "min_child_weight": cfg.min_child_weight,
        "lambda": cfg.reg_lambda,
        "gamma": cfg.gamma,
        "tree_method": cfg.tree_method,
        "device": cfg.device,
        "verbosity": 1,
        "seed": cfg.random_state,
    }

    oof = np.zeros(len(df_tr), dtype=np.float32)
    models, metrics = [], []
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y, groups=groups), 1):
        dtr = xgb.DMatrix(X[tr_idx], label=y[tr_idx], feature_names=features)
        dva = xgb.DMatrix(X[va_idx], label=y[va_idx], feature_names=features)
        bst = xgb.train(params=params, dtrain=dtr, num_boost_round=cfg.n_estimators,
                        evals=[(dva, "valid")], early_stopping_rounds=early_stopping_rounds,
                        verbose_eval=False)
        proba_va = _bst_predict_proba(bst, dva)
        oof[va_idx] = proba_va
        auc = roc_auc_score(y[va_idx], proba_va) if len(np.unique(y[va_idx]))>1 else np.nan
        ll  = float(log_loss(y[va_idx], np.clip(proba_va,1e-15,1-1e-15)))
        print(f"[MIN] Fold {fold} | AUC={auc:.5f} | Logloss={ll:.5f} | BestIter={getattr(bst,'best_iteration',None)}")
        bst.save_model(os.path.join(OUT_DIR, f"xgb_MIN_fold{fold}.json"))
        models.append(bst); metrics.append({"fold":fold,"auc":float(auc),"logloss":float(ll)})
        del dtr, dva; gc.collect()
    return {"oof":oof, "models":models, "metrics":metrics, "features":features}

groups = df_tr_s[group_key] if group_key in df_tr_s.columns else None
res_MIN = fit_xgb_cv_booster(df_tr_s, FEATURES, target_col, groups, cfg, "xgb_MIN")

# OOF 확률 + 임계치 탐색
df_tr_s["proba_min"] = res_MIN["oof"]
thr_min, f1_min = search_best_threshold_fast(
    df_tr_s[[order_key, product_col, target_col, "proba_min"]],
    order_key, product_col, target_col, "proba_min",
    n_candidates=31, max_orders=50_000, verbose=True
)
auc_min, ll_min = fast_auc_logloss(df_tr_s, target_col, "proba_min", max_rows=2_000_000)
print(f"[OOF-MIN] BestThr={thr_min:.4f} | F1={f1_min:.5f} | AUC≈{auc_min:.5f} | Logloss≈{ll_min:.5f}")

# OOF 저장
df_tr_s[[order_key, member_col, product_col, target_col, "proba_min"]].to_csv(OUT_OOF_PATH, index=False)
print("Saved OOF:", OUT_OOF_PATH)

# ---- 6) Test(6th) predict & save ----
def predict_with_boosters(df_in: pd.DataFrame, features: List[str], models: List[xgb.Booster], out_col: str):
    X = df_in[features].values.astype(np.float32)
    dmat = xgb.DMatrix(X, feature_names=features)
    preds = np.zeros(len(df_in), dtype=np.float32)
    for bst in models:
        preds += _bst_predict_proba(bst, dmat)
    preds /= max(1, len(models))
    df_in[out_col] = preds
    return preds

predict_with_boosters(df_test, res_MIN["features"], res_MIN["models"], "proba_min")

test_auc_min, test_ll_min = fast_auc_logloss(df_test, target_col, "proba_min", max_rows=2_000_000)
test_f1_min = order_level_f1(df_test, order_key, product_col, target_col, "proba_min", thr_min)
print(f"[TEST-MIN] Thr={thr_min:.4f} | F1={test_f1_min:.5f} | AUC≈{test_auc_min:.5f} | Logloss≈{test_ll_min:.5f}")

# 주문별 제출 포맷
def to_pred_string(g: pd.DataFrame, thr: float) -> str:
    items = g.loc[g["proba_min"] >= thr, product_col].astype(str).tolist()
    return "None" if len(items)==0 else " ".join(items)

sub_series = df_test.groupby(order_key, group_keys=False).apply(lambda g: to_pred_string(g, thr_min))
sub_df = sub_series.reset_index()
sub_df.columns = [order_key, "products"]
ord_path = f"{OUT_ORD_TEST}_{thr_min:.2f}.csv"
sub_df.to_csv(ord_path, index=False)

# 상세 확률 저장
df_test[[order_key, member_col, product_col, target_col, "proba_min"]].to_csv(OUT_DET_TEST, index=False)

print("Saved TEST:")
print(" - detailed:", OUT_DET_TEST)
print(" - orders  :", ord_path)

# ---- 7) Feature importance ----
def save_importance_booster(models: List[xgb.Booster], features: List[str], out_path: str):
    agg = np.zeros(len(features), dtype=np.float64)
    for bst in models:
        fmap = bst.get_score(importance_type='gain')
        for i, f in enumerate(features):
            v = fmap.get(f, 0.0)
            if v == 0.0: v = fmap.get(f"f{i}", 0.0)
            agg[i] += float(v or 0.0)
    agg /= max(1, len(models))
    pd.DataFrame({"feature": features, "gain": agg}).sort_values("gain", ascending=False).to_csv(out_path, index=False)
    print("Saved feature importance:", out_path)

save_importance_booster(res_MIN['models'], res_MIN['features'], OUT_FEAT_IMP)

# ---- 8) Summary & Meta ----
summary_rows = [{
    "model": "XGB_minimal_5feats",
    "features_used": res_MIN["features"],
    "macro_F1_test": float(test_f1_min),
    "AUC_test": float(test_auc_min),
    "Logloss_test": float(test_ll_min),
    "best_threshold_from_oof": float(thr_min),
    "train_rows_used": int(len(df_tr_s)),
    "pos_in_train_sample": int((df_tr_s[target_col]==1).sum())
}]
pd.DataFrame(summary_rows).to_csv(OUT_SUMMARY_CSV, index=False)
with open(OUT_SUMMARY_JSON, "w") as f:
    json.dump({"summary": summary_rows,
               "notes": {
                   "master_path": MASTER_PATH,
                   "columns": {
                       "target": target_col,
                       "order": order_col,
                       "user": member_col,
                       "product": product_col,
                       "hour": hour_col,
                       "aisle_id": aisle_col,
                       "department_id": dept_col
                   },
                   "artifacts": {
                       "oof": OUT_OOF_PATH,
                       "detailed": OUT_DET_TEST,
                       "orders": ord_path,
                       "feature_importance": OUT_FEAT_IMP
                   }
               }}, f, indent=2, ensure_ascii=False)

print("\n===== DONE (Minimal features) =====")
print(pd.DataFrame(summary_rows))
print("\nArtifacts:")
print(" - OOF:", OUT_OOF_PATH)
print(" - Detailed:", OUT_DET_TEST)
print(" - Orders:", ord_path)
print(" - Feature importance:", OUT_FEAT_IMP)
print(" - Summary CSV:", OUT_SUMMARY_CSV)
print(" - Summary JSON:", OUT_SUMMARY_JSON)
print(" - Meta JSON:", OUT_META_PATH)

# 메타 별도 저장(구성/지표/모델 파일)
with open(OUT_META_PATH, "w") as f:
    json.dump({
        "config": {
            "NFOLDS": NFOLDS,
            "EARLY_STOP": EARLY_STOP,
            "NEG_POS_RATIO": NEG_POS_RATIO,
            "MAX_TRAIN_ROWS": MAX_TRAIN_ROWS,
            "DEVICE": DEVICE,
            "TREE_METHOD": "hist",
            "SEED": 42
        },
        "features": res_MIN["features"],
        "metrics": {
            "oof": {"best_thr": float(thr_min), "f1": float(f1_min), "auc": float(auc_min), "logloss": float(ll_min)},
            "test": {"f1": float(test_f1_min), "auc": float(test_auc_min), "logloss": float(test_ll_min)}
        },
        "artifacts": {
            "oof_predictions": os.path.basename(OUT_OOF_PATH),
            "test_predictions_detailed": os.path.basename(OUT_DET_TEST),
            "test_predictions_orders": os.path.basename(ord_path),
            "feature_importance": os.path.basename(OUT_FEAT_IMP),
            "models": [f"xgb_MIN_fold{i}.json" for i in range(1, NFOLDS+1)],
            "summary_csv": os.path.basename(OUT_SUMMARY_CSV),
            "summary_json": os.path.basename(OUT_SUMMARY_JSON)
        }
    }, f, indent=2, ensure_ascii=False)


Mounted at /content/drive
[INFO] device=cuda  tree_method=hist  xgboost=3.0.4
eval_set=='test' 행 11,792,498개 제거.
[INFO] rows — train(prior)=20,641,991, test(6th)=1,384,617
[INFO] minimal features: ['order_hour_of_day', 'aisle_id', 'department_id', 'product_id', 'user_id']
[INFO] train sample: 3,000,000 rows (pos=1,768,007)
[MIN] Fold 1 | AUC=0.64770 | Logloss=0.64113 | BestIter=593
[MIN] Fold 2 | AUC=0.64880 | Logloss=0.64110 | BestIter=596
[MIN] Fold 3 | AUC=0.64975 | Logloss=0.64094 | BestIter=590
[thr-search] sample orders: 50,000  rows: 107,930
[thr-search] candidates: 31  (quantiles 0.05~0.95)
[thr-search] 6/31  thr=0.5045  F1=0.63315  elapsed=0.4s
[thr-search] 12/31  thr=0.5658  F1=0.60015  elapsed=0.5s
[thr-search] 18/31  thr=0.6094  F1=0.54773  elapsed=0.6s
[thr-search] 24/31  thr=0.6663  F1=0.47327  elapsed=0.6s
[thr-search] 30/31  thr=0.7565  F1=0.36293  elapsed=0.7s
[thr-search] 31/31  thr=0.7807  F1=0.33886  elapsed=0.7s
[thr-search] best_thr=0.4334  best_f1=0.64142
[metric

/tmp/ipython-input-1622234830.py:334: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_series = df_test.groupby(order_key, group_keys=False).apply(lambda g: to_pred_string(g, thr_min))


Saved TEST:
 - detailed: /content/drive/MyDrive/data/instacart/perf_minimal_only6/test_predictions_detailed_minimal.csv
 - orders  : /content/drive/MyDrive/data/instacart/perf_minimal_only6/test_predictions_orders_minimal_threshold_0.43.csv
Saved feature importance: /content/drive/MyDrive/data/instacart/perf_minimal_only6/xgb_minimal_feature_importance.csv

===== DONE (Minimal features) =====
                model                                      features_used  \
0  XGB_minimal_5feats  [order_hour_of_day, aisle_id, department_id, p...   

   macro_F1_test  AUC_test  Logloss_test  best_threshold_from_oof  \
0       0.696575  0.651927       0.63678                 0.433414   

   train_rows_used  pos_in_train_sample  
0          3000000              1768007  

Artifacts:
 - OOF: /content/drive/MyDrive/data/instacart/perf_minimal_only6/oof_predictions_instacart_minimal.csv
 - Detailed: /content/drive/MyDrive/data/instacart/perf_minimal_only6/test_predictions_detailed_minimal.csv
 - Or